# LeetCode 3264 — Final Array State After K Multiplication Operations I bằng Python

Link đề: https://leetcode.com/problems/final-array-state-after-k-multiplication-operations-i/ · Độ khó: **Easy** · Chủ đề: Array, Math, Heap (Priority Queue), Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc. Mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả đối chiếu hàng nghìn test ngẫu nhiên với bản mô phỏng vét cạn).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng `nums`, số lần `k` và hệ số `multiplier`; thực hiện `k` thao tác, mỗi thao tác **tìm giá trị nhỏ nhất** `x` trong `nums` (nếu nhiều phần tử cùng nhỏ nhất thì chọn cái **xuất hiện đầu tiên**) rồi **thay `x` bằng `x * multiplier`**; trả về mảng cuối cùng.
2. Nắm các công cụ Python: `min`, `list.index`, **gán lại theo chỉ số** `nums[i] = ...`, **sao chép list** (`nums[:]`, `list(nums)`) để tránh sửa nhầm đầu vào, **so sánh tuple** theo từ điển `(giá trị, chỉ số)`, và **`heapq`** (`heapify`, `heappop`, `heappush`) cùng **`bisect.insort`**.
3. Đi từ **quét tìm min bằng vòng lặp tay** (`O(k·n)`) → **`nums.index(min(nums))`** → **min-heap các cặp `(giá trị, chỉ số)`** (`O(n + k log n)`) → **danh sách có thứ tự + `insort`**.
4. Hiểu mấu chốt: bài này là **mô phỏng**; điểm tinh tế là **quy tắc "xuất hiện đầu tiên"** khi hòa — ta mã hóa nó bằng cách so sánh **cặp `(giá trị, chỉ số)`** để chỉ số nhỏ hơn thắng; và phần tử nhân lên **có thể vẫn là nhỏ nhất** ở lượt sau (khi `multiplier = 1` hoặc các số nhỏ).
5. Tránh các lỗi kinh điển: tính `min * multiplier` mà **quên gán lại** vào mảng, **sắp xếp** mảng làm mất vị trí, lấy **`max`** thay vì `min`, quên **đẩy lại** phần tử mới vào heap, và dùng cặp `(chỉ số, giá trị)` sai thứ tự khi so sánh.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, một số nguyên `k` và một số nguyên `multiplier`.

Bạn cần thực hiện **`k` thao tác** trên `nums`. Trong mỗi thao tác:

- Tìm giá trị **nhỏ nhất** `x` trong `nums`. Nếu có nhiều phần tử cùng bằng giá trị nhỏ nhất, chọn phần tử **xuất hiện đầu tiên** (chỉ số nhỏ nhất).
- Thay phần tử `x` đã chọn bằng `x * multiplier`.

Hãy trả về mảng số nguyên thể hiện **trạng thái cuối cùng** của `nums` sau khi thực hiện đủ `k` thao tác.

**Ví dụ 1**

- Input: `nums = [2,1,3,5,6]`, `k = 5`, `multiplier = 2`
- Output: `[8,4,6,5,6]`
- Giải thích: `[2,1,3,5,6]` → `[2,2,3,5,6]` → `[4,2,3,5,6]` → `[4,4,3,5,6]` → `[4,4,6,5,6]` → `[8,4,6,5,6]`.

**Ví dụ 2**

- Input: `nums = [1,2]`, `k = 3`, `multiplier = 4`
- Output: `[16,8]`
- Giải thích: `[1,2]` → `[4,2]` → `[4,8]` → `[16,8]`.

**Ràng buộc:** `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`; `1 <= k <= 10`; `1 <= multiplier <= 5`.

In [1]:
from typing import List
import heapq
from bisect import insort
import random

examples = [
    ([2, 1, 3, 5, 6], 5, 2, [8, 4, 6, 5, 6]),
    ([1, 2], 3, 4, [16, 8]),
]
for nums, k, multiplier, expected in examples:
    print(f"nums={nums}, k={k}, multiplier={multiplier} -> kỳ vọng {expected}")

nums=[2, 1, 3, 5, 6], k=5, multiplier=2 -> kỳ vọng [8, 4, 6, 5, 6]
nums=[1, 2], k=3, multiplier=4 -> kỳ vọng [16, 8]


## 3. Kiến thức nền

### 3.1. Chỉ số và gán lại theo chỉ số

`nums[i]` là phần tử ở vị trí `i` (đếm từ `0`). Ta có thể **đọc** `nums[i]` và **ghi** `nums[i] = giá_trị_mới` — thay đổi nằm ngay trong list. `nums[i] *= 3` là viết tắt của `nums[i] = nums[i] * 3`.

In [2]:
nums = [10, 20, 30]
print(nums[1])           # 20
nums[1] = 25
nums[2] *= 3             # 90
print(nums)              # [10, 25, 90]

20
[10, 25, 90]


### 3.2. Sao chép list: `nums[:]` khác `nums`

`b = a` **không** tạo list mới: `a` và `b` là **hai tên của cùng một list**, nên sửa qua `b` sẽ thấy ở `a`. Để có bản sao độc lập dùng `a[:]`, `list(a)` hoặc `a.copy()`. Điều này quan trọng vì bản nộp LeetCode sửa trực tiếp `nums`; còn khi test ta nên truyền **bản sao** để giữ lại đầu vào gốc.

In [3]:
a = [1, 2, 3]
b = a            # cùng một list
c = a[:]         # bản sao
b[0] = 99
print(a, b, c)   # [99, 2, 3] [99, 2, 3] [1, 2, 3]
print(a is b, a is c)    # True False

[99, 2, 3] [99, 2, 3] [1, 2, 3]
True False


### 3.3. `min` và `list.index`

`min(nums)` cho **giá trị** nhỏ nhất. `nums.index(x)` cho **chỉ số của lần xuất hiện đầu tiên** của `x` (báo `ValueError` nếu không có). Ghép lại: `nums.index(min(nums))` chính là **chỉ số của phần tử nhỏ nhất xuất hiện đầu tiên** — đúng yêu cầu của đề. Mỗi lệnh quét list nên tốn `O(n)`.

In [4]:
nums = [3, 1, 2, 1]
print(min(nums))                  # 1
print(nums.index(1))              # 1  (lần xuất hiện đầu tiên)
print(nums.index(min(nums)))      # 1

1
1
1


### 3.4. So sánh tuple theo từ điển và `min(..., key=...)`

Python so sánh hai tuple **từng phần tử từ trái sang phải**: `(1, 5) < (2, 0)` vì `1 < 2`; nếu phần tử đầu bằng nhau thì xét tiếp phần tử sau: `(2, 0) < (2, 3)`. Nhờ vậy cặp `(giá trị, chỉ số)` tự động: **giá trị nhỏ hơn thắng; hòa thì chỉ số nhỏ hơn thắng** — chính là quy tắc đề bài. `min(range(n), key=nums.__getitem__)` cũng trả chỉ số của phần tử nhỏ nhất (đầu tiên khi hòa).

In [5]:
print((1, 5) < (2, 0), (2, 0) < (2, 3), (2, 3) < (2, 0))     # True True False
nums = [4, 2, 2, 7]
pairs = [(x, i) for i, x in enumerate(nums)]
print(min(pairs))                                    # (2, 1)
print(min(range(len(nums)), key=lambda i: nums[i]))  # 1 (hòa -> chỉ số đầu tiên)

True True False
(2, 1)
1


### 3.5. `enumerate` để lấy cả chỉ số và giá trị

`enumerate(nums)` cho cặp `(i, x)` theo thứ tự. Dùng để tạo cặp `(x, i)` hoặc để quét tìm min kèm chỉ số.

In [6]:
for i, x in enumerate(["a", "b", "c"]):
    print(i, x)
print([(x, i) for i, x in enumerate([5, 6, 7])])    # [(5, 0), (6, 1), (7, 2)]

0 a
1 b
2 c
[(5, 0), (6, 1), (7, 2)]


### 3.6. Min-heap với `heapq`

**Min-heap** (đống nhỏ nhất) là cấu trúc cho phép lấy **phần tử nhỏ nhất** rất nhanh. Trong Python, `heapq` biến một **list thường** thành heap:

- `heapq.heapify(lst)`: biến `lst` thành heap **tại chỗ** trong `O(n)`.
- `heapq.heappop(lst)`: lấy và bỏ phần tử **nhỏ nhất** (`O(log n)`).
- `heapq.heappush(lst, x)`: thêm `x` (`O(log n)`).
- `lst[0]` luôn là phần tử nhỏ nhất (xem mà không bỏ).

Nó so sánh các phần tử bằng `<`, nên với tuple nó dùng thứ tự từ điển ở 3.4.

In [7]:
h = [(5, 0), (2, 1), (9, 2), (2, 3)]
heapq.heapify(h)
print(h[0])                         # (2, 1): nhỏ nhất, hòa giá trị 2 thì chỉ số 1 < 3
print(heapq.heappop(h))             # (2, 1)
heapq.heappush(h, (1, 7))
print(heapq.heappop(h), heapq.heappop(h))     # (1, 7) (2, 3)

(2, 1)
(2, 1)
(1, 7) (2, 3)


### 3.7. Danh sách luôn được sắp: `bisect.insort`

`insort(lst, x)` chèn `x` vào list **đã sắp tăng** sao cho list vẫn sắp tăng (tìm chỗ bằng tìm kiếm nhị phân `O(log n)`, nhưng **dịch phần tử** tốn `O(n)`). Với tuple `(giá trị, chỉ số)` nó cũng so theo từ điển. Phần tử nhỏ nhất luôn ở `lst[0]` (`lst.pop(0)` lấy ra).

In [8]:
sorted_list = [(1, 0), (3, 2), (6, 1)]
insort(sorted_list, (3, 1))
print(sorted_list)                  # [(1, 0), (3, 1), (3, 2), (6, 1)]
print(sorted_list.pop(0), sorted_list)

[(1, 0), (3, 1), (3, 2), (6, 1)]
(1, 0) [(3, 1), (3, 2), (6, 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm Ví dụ 1 bằng tay: `nums = [2, 1, 3, 5, 6]`, `k = 5`, `multiplier = 2`.

Mỗi lần: **nhìn cả dãy, tìm số nhỏ nhất (gần trái nhất), nhân đôi nó tại chỗ**.

| thao tác | mảng trước | nhỏ nhất (chỉ số) | mảng sau |
|---|---|---|---|
| 1 | `[2,1,3,5,6]` | `1` (chỉ số 1) | `[2,2,3,5,6]` |
| 2 | `[2,2,3,5,6]` | `2` (chỉ số 0, hòa với chỉ số 1) | `[4,2,3,5,6]` |
| 3 | `[4,2,3,5,6]` | `2` (chỉ số 1) | `[4,4,3,5,6]` |
| 4 | `[4,4,3,5,6]` | `3` (chỉ số 2) | `[4,4,6,5,6]` |
| 5 | `[4,4,6,5,6]` | `4` (chỉ số 0, hòa với chỉ số 1) | `[8,4,6,5,6]` |

Hai điều rút ra:

1. **Hòa phải chọn chỉ số nhỏ hơn** (thao tác 2 và 5) — nếu chọn sai, kết quả khác (`[4,4,...]` hoặc `[2,4,...]`).
2. Phần tử vừa nhân **có thể lại là nhỏ nhất** ở lượt sau: ở thao tác 3 ta chọn lại chỉ số `1`, chính là phần tử mang giá trị `2` vừa tạo ra ở thao tác 1. Mỗi chỉ số có thể được nhân nhiều lần.

In [9]:
nums = [2, 1, 3, 5, 6]
for step in range(1, 6):
    x = min(nums)
    i = nums.index(x)
    nums[i] = x * 2
    print(f"thao tác {step}: chọn {x} ở chỉ số {i} -> {nums}")

thao tác 1: chọn 1 ở chỉ số 1 -> [2, 2, 3, 5, 6]
thao tác 2: chọn 2 ở chỉ số 0 -> [4, 2, 3, 5, 6]
thao tác 3: chọn 2 ở chỉ số 1 -> [4, 4, 3, 5, 6]
thao tác 4: chọn 3 ở chỉ số 2 -> [4, 4, 6, 5, 6]
thao tác 5: chọn 4 ở chỉ số 0 -> [8, 4, 6, 5, 6]


## 5. Cách 1 — Brute Force / baseline: quét tay tìm min rồi nhân

**Ý tưởng.** Làm đúng như đề: lặp `k` lần; mỗi lần dùng **vòng `for` quét cả mảng** giữ `best` là **chỉ số** của phần tử nhỏ nhất hiện tại. Để chọn "đầu tiên khi hòa" ta chỉ cập nhật `best` khi gặp giá trị **nhỏ hơn hẳn** (`<`, không phải `<=`).

In [10]:
def get_final_state_brute(nums: List[int], k: int, multiplier: int) -> List[int]:
    nums = nums[:]                         # không sửa đầu vào
    n = len(nums)
    for _ in range(k):
        best = 0
        for i in range(1, n):
            if nums[i] < nums[best]:       # "<" chứ không "<=": hòa thì giữ chỉ số nhỏ hơn
                best = i
        nums[best] *= multiplier
    return nums


for nums, k, multiplier, expected in examples:
    print(nums, k, multiplier, "->", get_final_state_brute(nums, k, multiplier), "(kỳ vọng", expected, ")")

[2, 1, 3, 5, 6] 5 2 -> [8, 4, 6, 5, 6] (kỳ vọng [8, 4, 6, 5, 6] )
[1, 2] 3 4 -> [16, 8] (kỳ vọng [16, 8] )


**Độ phức tạp.** Mỗi thao tác quét `n` phần tử ⇒ `O(k · n)` time; `O(n)` space cho bản sao (hoặc `O(1)` nếu sửa thẳng đầu vào).

**Nút thắt.** Với `n <= 100`, `k <= 10` thì chỉ ~`1000` bước — quá nhanh. Nhưng mỗi thao tác ta **quét lại cả mảng** để tìm min; khi `k` và `n` lớn (bài II: `k` tới `10^9`) cần cấu trúc lấy min nhanh hơn như **heap**.

## 6. Ý tưởng thuật toán / bước đột phá

Cái ta lặp đi lặp lại là: **"lấy phần tử nhỏ nhất, rồi đưa phiên bản đã nhân vào lại"**. Đó đúng là việc của **hàng đợi ưu tiên (heap)**:

1. Mã hóa mỗi phần tử thành cặp **`(giá trị, chỉ số)`**. So sánh tuple cho đúng luật đề: nhỏ nhất theo giá trị, hòa thì chỉ số nhỏ.
2. `heapify` mọi cặp `(nums[i], i)` — `O(n)`.
3. `k` lần: `heappop` lấy `(x, i)` nhỏ nhất; ghi `nums[i] = x * multiplier`; `heappush` cặp mới `(nums[i], i)`.

Mỗi thao tác `O(log n)` thay vì `O(n)`. Chỉ số `i` đi kèm cặp giúp ta ghi lại **đúng vị trí** trong `nums`, và mảng `nums` chính là đáp án cuối.

(Cách 4 ở dưới dùng danh sách luôn sắp thứ tự `(giá trị, chỉ số)` để cùng ý tưởng nhưng với `insort`.)

In [11]:
# Cái heap làm: lấy cặp nhỏ nhất, nhân, đẩy lại — nhìn từng bước nhỏ
nums = [2, 1, 3]
heap = [(x, i) for i, x in enumerate(nums)]
heapq.heapify(heap)
print("heap ban đầu (phần tử đầu là nhỏ nhất):", heap)
x, i = heapq.heappop(heap)
print("lấy ra:", (x, i))
nums[i] = x * 2
heapq.heappush(heap, (nums[i], i))
print("nums =", nums, "| heap =", sorted(heap))

heap ban đầu (phần tử đầu là nhỏ nhất): [(1, 1), (2, 0), (3, 2)]
lấy ra: (1, 1)
nums = [2, 2, 3] | heap = [(2, 0), (2, 1), (3, 2)]


## 7. Các cách tiếp theo

Phần này có **3 cách** nữa (cùng Cách 1 là 4 cách). Mỗi cách: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### Cách 2 — Dùng `min` + `index` (gọn nhất bằng thư viện chuẩn)

Giải thích từng đoạn:

- Lặp `k` lần.
- `i = nums.index(min(nums))`: `min` tìm giá trị nhỏ nhất, `index` trả vị trí đầu tiên của nó — đúng luật "hòa thì đầu tiên".
- `nums[i] *= multiplier`: nhân tại chỗ.

In [12]:
def get_final_state_index(nums: List[int], k: int, multiplier: int) -> List[int]:
    nums = nums[:]
    for _ in range(k):
        i = nums.index(min(nums))
        nums[i] *= multiplier
    return nums


for nums, k, multiplier, expected in examples:
    assert get_final_state_index(nums, k, multiplier) == expected
print("Cách 2 chạy đúng trên các ví dụ")

Cách 2 chạy đúng trên các ví dụ


In [13]:
# Code nộp LeetCode — Cách 2
class Solution:
    def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]:
        for _ in range(k):
            i = nums.index(min(nums))
            nums[i] *= multiplier
        return nums


for nums, k, multiplier, expected in examples:
    assert Solution().getFinalState(nums[:], k, multiplier) == expected
print("OK")

OK


**Độ phức tạp Cách 2:** Time `O(k · n)` (`min` quét `n`, `index` quét tối đa `n`), Space `O(1)` thêm (sửa tại chỗ).

### Cách 3 — Min-heap các cặp `(giá trị, chỉ số)` (cách chuẩn)

Giải thích từng đoạn:

- `pq = [(x, i) for i, x in enumerate(nums)]` tạo cặp `(giá trị, chỉ số)`; `heapify(pq)`.
- Mỗi thao tác: `_, i = heappop(pq)` lấy cặp nhỏ nhất (ta chỉ cần chỉ số `i`).
- `nums[i] *= multiplier` cập nhật mảng đáp án.
- `heappush(pq, (nums[i], i))` đưa phiên bản mới của phần tử vào lại heap (**không được quên**, vì phần tử đó vẫn có thể là nhỏ nhất ở lượt sau).

In [14]:
def get_final_state_heap(nums: List[int], k: int, multiplier: int) -> List[int]:
    nums = nums[:]
    pq = [(x, i) for i, x in enumerate(nums)]
    heapq.heapify(pq)
    for _ in range(k):
        _, i = heapq.heappop(pq)
        nums[i] *= multiplier
        heapq.heappush(pq, (nums[i], i))
    return nums


for nums, k, multiplier, expected in examples:
    assert get_final_state_heap(nums, k, multiplier) == expected
print("Cách 3 chạy đúng trên các ví dụ")

Cách 3 chạy đúng trên các ví dụ


In [15]:
# Code nộp LeetCode — Cách 3
class Solution:
    def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]:
        pq = [(x, i) for i, x in enumerate(nums)]
        heapq.heapify(pq)
        for _ in range(k):
            _, i = heapq.heappop(pq)
            nums[i] *= multiplier
            heapq.heappush(pq, (nums[i], i))
        return nums


for nums, k, multiplier, expected in examples:
    assert Solution().getFinalState(nums[:], k, multiplier) == expected
print("OK")

OK


**Độ phức tạp Cách 3:** Time `O(n + k log n)` (`heapify` `O(n)`, mỗi thao tác một `pop` + một `push` `O(log n)`), Space `O(n)` cho heap. Có thể dùng `heapq.heapreplace(pq, new)` (pop rồi push trong một lần) nhưng phải biết `i` trước — cách trên rõ ràng hơn cho người mới.

### Cách 4 — Danh sách luôn sắp `(giá trị, chỉ số)` + `insort`

Giải thích từng đoạn:

- `order = sorted((x, i) for i, x in enumerate(nums))`: danh sách cặp **sắp tăng** theo luật đề. Phần tử đầu `order[0]` là cái cần nhân.
- Mỗi thao tác: `x, i = order.pop(0)`; `nums[i] = x * multiplier`; `insort(order, (nums[i], i))` chèn lại đúng chỗ để list vẫn sắp tăng.
- Giống heap về ý tưởng nhưng thao tác trên list thường (đơn giản hơn để hình dung, chậm hơn một chút vì `pop(0)`/`insort` dịch phần tử `O(n)`).

In [16]:
def get_final_state_sorted(nums: List[int], k: int, multiplier: int) -> List[int]:
    nums = nums[:]
    order = sorted((x, i) for i, x in enumerate(nums))
    for _ in range(k):
        x, i = order.pop(0)
        nums[i] = x * multiplier
        insort(order, (nums[i], i))
    return nums


for nums, k, multiplier, expected in examples:
    assert get_final_state_sorted(nums, k, multiplier) == expected
print("Cách 4 chạy đúng trên các ví dụ")

Cách 4 chạy đúng trên các ví dụ


In [17]:
# Code nộp LeetCode — Cách 4
class Solution:
    def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]:
        order = sorted((x, i) for i, x in enumerate(nums))
        for _ in range(k):
            x, i = order.pop(0)
            nums[i] = x * multiplier
            insort(order, (nums[i], i))
        return nums


for nums, k, multiplier, expected in examples:
    assert Solution().getFinalState(nums[:], k, multiplier) == expected
print("OK")

OK


**Độ phức tạp Cách 4:** Time `O(n log n + k · n)` (sắp xếp ban đầu `O(n log n)`; mỗi thao tác `pop(0)` và `insort` dịch tối đa `O(n)`), Space `O(n)`. Với `n <= 100`, `k <= 10` không đáng kể; heap tốt hơn khi `k` lớn.

## 8. TRACE từng bước

Ta trace Cách 3 (heap) trên Ví dụ 1; Cách 2 (`index(min)`) trên ví dụ tự dựng `nums = [3, 3, 1, 2]`, `k = 4`, `multiplier = 3` (tự dựng; đáp án kiểm bằng vét cạn ngay trong cell); và Cách 4 (danh sách sắp) trên Ví dụ 2.

In [18]:
def trace_heap(nums: List[int], k: int, multiplier: int) -> List[int]:
    nums = nums[:]
    print(f"nums = {nums}, k = {k}, multiplier = {multiplier}")
    pq = [(x, i) for i, x in enumerate(nums)]
    heapq.heapify(pq)
    print("  heap ban đầu:", sorted(pq))
    for step in range(1, k + 1):
        x, i = heapq.heappop(pq)
        nums[i] = x * multiplier
        heapq.heappush(pq, (nums[i], i))
        print(f"  thao tác {step}: lấy {(x, i)}, nhân -> nums[{i}] = {nums[i]}, nums = {nums}, heap = {sorted(pq)}")
    return nums


assert trace_heap([2, 1, 3, 5, 6], 5, 2) == [8, 4, 6, 5, 6]

nums = [2, 1, 3, 5, 6], k = 5, multiplier = 2
  heap ban đầu: [(1, 1), (2, 0), (3, 2), (5, 3), (6, 4)]
  thao tác 1: lấy (1, 1), nhân -> nums[1] = 2, nums = [2, 2, 3, 5, 6], heap = [(2, 0), (2, 1), (3, 2), (5, 3), (6, 4)]
  thao tác 2: lấy (2, 0), nhân -> nums[0] = 4, nums = [4, 2, 3, 5, 6], heap = [(2, 1), (3, 2), (4, 0), (5, 3), (6, 4)]
  thao tác 3: lấy (2, 1), nhân -> nums[1] = 4, nums = [4, 4, 3, 5, 6], heap = [(3, 2), (4, 0), (4, 1), (5, 3), (6, 4)]
  thao tác 4: lấy (3, 2), nhân -> nums[2] = 6, nums = [4, 4, 6, 5, 6], heap = [(4, 0), (4, 1), (5, 3), (6, 2), (6, 4)]
  thao tác 5: lấy (4, 0), nhân -> nums[0] = 8, nums = [8, 4, 6, 5, 6], heap = [(4, 1), (5, 3), (6, 2), (6, 4), (8, 0)]


In [19]:
def trace_index(nums: List[int], k: int, multiplier: int) -> List[int]:
    nums = nums[:]
    print(f"nums = {nums}, k = {k}, multiplier = {multiplier}")
    for step in range(1, k + 1):
        x = min(nums)
        i = nums.index(x)
        nums[i] = x * multiplier
        print(f"  thao tác {step}: min = {x}, index = {i}, nums -> {nums}")
    return nums


res = trace_index([3, 3, 1, 2], 4, 3)
assert res == get_final_state_brute([3, 3, 1, 2], 4, 3)
print("kết quả:", res)

nums = [3, 3, 1, 2], k = 4, multiplier = 3
  thao tác 1: min = 1, index = 2, nums -> [3, 3, 3, 2]
  thao tác 2: min = 2, index = 3, nums -> [3, 3, 3, 6]
  thao tác 3: min = 3, index = 0, nums -> [9, 3, 3, 6]
  thao tác 4: min = 3, index = 1, nums -> [9, 9, 3, 6]
kết quả: [9, 9, 3, 6]


In [20]:
def trace_sorted(nums: List[int], k: int, multiplier: int) -> List[int]:
    nums = nums[:]
    order = sorted((x, i) for i, x in enumerate(nums))
    print(f"nums = {nums}, order = {order}")
    for step in range(1, k + 1):
        x, i = order.pop(0)
        nums[i] = x * multiplier
        insort(order, (nums[i], i))
        print(f"  thao tác {step}: pop {(x, i)} -> nums = {nums}, order = {order}")
    return nums


assert trace_sorted([1, 2], 3, 4) == [16, 8]

nums = [1, 2], order = [(1, 0), (2, 1)]
  thao tác 1: pop (1, 0) -> nums = [4, 2], order = [(2, 1), (4, 0)]
  thao tác 2: pop (2, 1) -> nums = [4, 8], order = [(4, 0), (8, 1)]
  thao tác 3: pop (4, 0) -> nums = [16, 8], order = [(8, 1), (16, 0)]


Bảng trace Cách 3 với Ví dụ 1 (`nums = [2,1,3,5,6]`, `k = 5`, `multiplier = 2`):

| thao tác | pop `(x, i)` | `nums` sau | cặp đẩy lại |
|---|---|---|---|
| 1 | (1, 1) | `[2,2,3,5,6]` | (2, 1) |
| 2 | (2, 0) | `[4,2,3,5,6]` | (4, 0) |
| 3 | (2, 1) | `[4,4,3,5,6]` | (4, 1) |
| 4 | (3, 2) | `[4,4,6,5,6]` | (6, 2) |
| 5 | (4, 0) | `[8,4,6,5,6]` | (8, 0) |

Ở thao tác 2, heap chứa cả `(2, 0)` và `(2, 1)`; so sánh tuple chọn `(2, 0)` vì `0 < 1` — đó là cách luật "đầu tiên khi hòa" được hiện thực hóa. Cùng quy tắc ở thao tác 5 với `(4, 0)` và `(4, 1)`.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3 (heap)**: chuẩn, nhanh và mở rộng tốt khi `k` lớn. (Ghi chú: LeetCode đã `import heapq` / có sẵn `heapify`... ở môi trường chấm; trong notebook ta đã `import heapq` ở đầu.)

In [21]:
class Solution:
    def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]:
        pq = [(x, i) for i, x in enumerate(nums)]   # cặp (giá trị, chỉ số): hòa thì chỉ số nhỏ hơn thắng
        heapq.heapify(pq)
        for _ in range(k):
            _, i = heapq.heappop(pq)                # lấy phần tử nhỏ nhất (đầu tiên khi hòa)
            nums[i] *= multiplier                   # nhân tại chỗ
            heapq.heappush(pq, (nums[i], i))        # đẩy phiên bản mới vào lại heap
        return nums


sol = Solution()
for nums, k, multiplier, expected in examples:
    print(nums, k, multiplier, "->", sol.getFinalState(nums[:], k, multiplier), "(kỳ vọng", expected, ")")

[2, 1, 3, 5, 6] 5 2 -> [8, 4, 6, 5, 6] (kỳ vọng [8, 4, 6, 5, 6] )
[1, 2] 3 4 -> [16, 8] (kỳ vọng [16, 8] )


## 10. Vì sao thuật toán đúng

- **Bất biến.** Trước mỗi thao tác, `pq` chứa **đúng** một cặp `(nums[i], i)` cho **mỗi** chỉ số `i` với giá trị **hiện tại** của `nums[i]`. Ban đầu đúng nhờ khởi tạo; mỗi thao tác ta `pop` cặp cũ của chỉ số `i` rồi `push` cặp mới của chính `i`, nên bất biến được giữ.
- **Chọn đúng phần tử.** `heappop` trả cặp **nhỏ nhất theo thứ tự từ điển** `(giá trị, chỉ số)`, nghĩa là giá trị nhỏ nhất, và trong các phần tử cùng giá trị nhỏ nhất thì **chỉ số nhỏ nhất** (xuất hiện đầu tiên) — khớp từng chữ của đề.
- **Ghi đúng vị trí.** Cặp mang theo chỉ số `i`, nên `nums[i] *= multiplier` sửa đúng phần tử đã chọn; các phần tử khác không đổi.
- **Đủ `k` thao tác.** Vòng lặp chạy đúng `k` lần, mỗi lần một thao tác như đề; trạng thái cuối chính là `nums`.
- **Trường hợp `multiplier = 1`:** phần tử nhỏ nhất không đổi, nhưng vẫn là nhỏ nhất, thuật toán cứ lặp lại thao tác không làm đổi mảng — kết quả vẫn đúng (mảng giữ nguyên).
- **Đối chiếu độc lập.** Cách 1 (quét tay theo định nghĩa) được so với các cách còn lại trên hàng nghìn test ngẫu nhiên ở mục 12.

## 11. Time / Space Complexity

Gọi `n = len(nums)`, `k` là số thao tác.

| Cách | Time | Space (thêm) |
|---|---|---|
| 1. Quét tay | `O(k · n)` | `O(1)` (sửa tại chỗ) |
| 2. `index(min)` | `O(k · n)` | `O(1)` |
| 3. Min-heap | `O(n + k log n)` | `O(n)` |
| 4. List sắp + `insort` | `O(n log n + k · n)` | `O(n)` |

Với `n <= 100`, `k <= 10` mọi cách chạy tức thì. Heap là cách thích hợp nhất khi `k` lớn (hoặc `n` lớn).

Lưu ý: giá trị có thể tăng tới `100 * 5^10 ≈ 10^9` — Python không lo tràn số (ở ngôn ngữ khác cần `long`).

## 12. Bộ test

Gồm: ví dụ đề, biên (một phần tử, `multiplier = 1`, tất cả bằng nhau, `k` lớn nhất), kiểm tra luật hòa, đối chiếu cả `4` cách và ngẫu nhiên có seed đúng ràng buộc (`1 <= n <= 100`, `1 <= nums[i] <= 100`, `1 <= k <= 10`, `1 <= multiplier <= 5`).

In [22]:
IMPLS = {
    "brute": get_final_state_brute,
    "index": get_final_state_index,
    "heap": get_final_state_heap,
    "sorted": get_final_state_sorted,
    "Solution": lambda nums, k, m: Solution().getFinalState(nums[:], k, m),
}

# 1) ví dụ đề
for nums, k, multiplier, expected in examples:
    for name, f in IMPLS.items():
        assert f(nums, k, multiplier) == expected, (name, nums, k, multiplier)

# 2) biên và tự dựng (đáp án tính tay)
manual = [
    ([5], 3, 2, [40]),                              # một phần tử: nhân 3 lần
    ([1, 2, 3], 4, 1, [1, 2, 3]),                   # multiplier = 1: không đổi
    ([7, 7, 7], 3, 2, [14, 14, 14]),                # bằng nhau: lần lượt chỉ số 0, 1, 2
    ([7, 7, 7], 1, 3, [21, 7, 7]),                  # hòa: chọn chỉ số đầu tiên
    ([1, 100], 3, 5, [125, 100]),                   # 1 -> 5 -> 25 -> 125 (ba lần đều chọn chỉ số 0 vì 1, 5, 25 đều < 100)
]
for nums, k, m, expected in manual:
    for name, f in IMPLS.items():
        assert f(nums, k, m) == expected, (name, nums, k, m, f(nums, k, m), expected)

# kiểm đầu vào không bị sửa bởi các hàm demo (trừ bản nộp ghi thẳng, ta đã truyền bản sao)
original = [4, 1, 3]
copy_in = original[:]
for f in IMPLS.values():
    f(copy_in, 5, 2)
assert copy_in == original
print("ví dụ đề + biên: PASS")

ví dụ đề + biên: PASS


In [23]:
# 3) ngẫu nhiên có seed: đúng ràng buộc, mọi cách phải đồng nhất với vét cạn
rng = random.Random(3264)
for _ in range(4000):
    n = rng.randint(1, 100)
    hi = rng.choice([2, 5, 100])                 # nhiều giá trị lặp (hòa) khi hi nhỏ
    nums = [rng.randint(1, hi) for _ in range(n)]
    k = rng.randint(1, 10)
    m = rng.randint(1, 5)
    expected = get_final_state_brute(nums, k, m)
    for name, f in IMPLS.items():
        assert f(nums, k, m) == expected, (name, nums, k, m)
print("4000 test ngẫu nhiên: tất cả các cách đồng nhất — PASS")

# 4) kiểm tra độc lập theo định nghĩa (viết riêng, không dùng index/heap): mô phỏng bằng sorted có key
def reference(nums, k, m):
    a = list(nums)
    for _ in range(k):
        i = sorted(range(len(a)), key=lambda j: (a[j], j))[0]
        a[i] *= m
    return a


for _ in range(1000):
    n = rng.randint(1, 30)
    nums = [rng.randint(1, 6) for _ in range(n)]
    k, m = rng.randint(1, 10), rng.randint(1, 5)
    assert get_final_state_heap(nums, k, m) == reference(nums, k, m)
print("đối chiếu với bản tham chiếu độc lập: PASS")

4000 test ngẫu nhiên: tất cả các cách đồng nhất — PASS
đối chiếu với bản tham chiếu độc lập: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tính `min * multiplier` nhưng quên **gán lại**

Biểu thức `min(nums) * multiplier` chỉ tạo một **số mới**, không ghi vào mảng. Mảng giữ nguyên, kết quả sai.

In [24]:
def sai_quen_gan(nums, k, multiplier):
    nums = nums[:]
    for _ in range(k):
        x = min(nums)
        x * multiplier                  # SAI: tính xong rồi bỏ đi, không ghi vào nums
    return nums

print("sai:", sai_quen_gan([2, 1, 3, 5, 6], 5, 2), "(đúng là [8, 4, 6, 5, 6])")
print("đúng:", get_final_state_index([2, 1, 3, 5, 6], 5, 2))

sai: [2, 1, 3, 5, 6] (đúng là [8, 4, 6, 5, 6])
đúng: [8, 4, 6, 5, 6]


### Lỗi 2: Sắp xếp mảng (làm mất vị trí)

Đề yêu cầu trạng thái cuối **theo vị trí gốc**. Sắp xếp `nums` trước rồi nhân `k` phần tử đầu làm đổi thứ tự — sai. (Ngoài ra, phần tử vừa nhân có thể vẫn nhỏ nhất, nên "nhân `k` phần tử nhỏ nhất đầu tiên, mỗi cái một lần" cũng sai.)

In [25]:
def sai_sap_xep(nums, k, multiplier):
    nums = sorted(nums)                 # SAI: mất vị trí gốc
    for i in range(k):
        nums[i] *= multiplier           # và mỗi phần tử chỉ nhân một lần
    return nums

print("sai:", sai_sap_xep([2, 1, 3, 5, 6], 5, 2), "(đúng là [8, 4, 6, 5, 6])")
try:
    sai_sap_xep([1, 2], 3, 4)
except IndexError as e:
    print("IndexError:", e)

sai: [2, 4, 6, 10, 12] (đúng là [8, 4, 6, 5, 6])
IndexError: list index out of range


### Lỗi 3: Lấy `max` thay vì `min` (hoặc nhầm hướng so sánh)

Đề nhân phần tử **nhỏ nhất**. Đổi nhầm thành `max` làm phần tử lớn càng lớn; sai ở hầu hết ví dụ.

In [26]:
def sai_max(nums, k, multiplier):
    nums = nums[:]
    for _ in range(k):
        i = nums.index(max(nums))       # SAI: phải là min
        nums[i] *= multiplier
    return nums

print("sai:", sai_max([2, 1, 3, 5, 6], 5, 2), "(đúng là [8, 4, 6, 5, 6])")

sai: [2, 1, 3, 5, 192] (đúng là [8, 4, 6, 5, 6])


### Lỗi 4: Quên đẩy phần tử mới vào lại heap

Sau khi `heappop`, cặp cũ **đã rời heap**. Nếu không `heappush` cặp mới, phần tử đó không còn có cơ hội làm nhỏ nhất ở các lượt sau; heap dần cạn (có thể báo `IndexError: index out of range` khi `k > n`).

In [27]:
def sai_quen_push(nums, k, multiplier):
    nums = nums[:]
    pq = [(x, i) for i, x in enumerate(nums)]
    heapq.heapify(pq)
    for _ in range(k):
        _, i = heapq.heappop(pq)
        nums[i] *= multiplier
        # SAI: thiếu heappush(pq, (nums[i], i))
    return nums

print("sai (k <= n, tình cờ chạy được):", sai_quen_push([2, 1, 3, 5, 6], 5, 2), "vs đúng [8, 4, 6, 5, 6]")
try:
    sai_quen_push([1, 2], 3, 4)
except IndexError as e:
    print("sai khi k > n -> IndexError:", e)

sai (k <= n, tình cờ chạy được): [4, 2, 6, 10, 12] vs đúng [8, 4, 6, 5, 6]
sai khi k > n -> IndexError: index out of range


### Lỗi 5: Cặp `(chỉ số, giá trị)` sai thứ tự

Tuple so sánh **phần tử đầu trước**. Nếu viết `(i, x)` thì heap sẽ luôn lấy **chỉ số nhỏ nhất** (tức phần tử đầu mảng) bất kể giá trị — hoàn toàn sai.

In [28]:
def sai_thu_tu_cap(nums, k, multiplier):
    nums = nums[:]
    pq = [(i, x) for i, x in enumerate(nums)]     # SAI: phải là (x, i)
    heapq.heapify(pq)
    for _ in range(k):
        i, _ = heapq.heappop(pq)
        nums[i] *= multiplier
        heapq.heappush(pq, (i, nums[i]))
    return nums

print("sai:", sai_thu_tu_cap([2, 1, 3, 5, 6], 5, 2), "(đúng là [8, 4, 6, 5, 6])")

sai: [64, 1, 3, 5, 6] (đúng là [8, 4, 6, 5, 6])


### Lỗi 6: Dùng `<=` khi quét tay tìm min (chọn nhầm khi hòa)

Với `if nums[i] <= nums[best]` mỗi lần hòa ta **dời** `best` sang chỉ số lớn hơn, nên chọn phần tử **cuối cùng** thay vì **đầu tiên**.

In [29]:
def sai_le(nums, k, multiplier):
    nums = nums[:]
    n = len(nums)
    for _ in range(k):
        best = 0
        for i in range(1, n):
            if nums[i] <= nums[best]:     # SAI: "<=" chọn phần tử hòa cuối cùng
                best = i
        nums[best] *= multiplier
    return nums

print("sai:", sai_le([2, 1, 3, 5, 6], 5, 2), "(đúng là [8, 4, 6, 5, 6])")

sai: [4, 8, 6, 5, 6] (đúng là [8, 4, 6, 5, 6])


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Lặp `k` lần: lấy phần tử nhỏ nhất, sửa, đưa lại"** ⇒ nghĩ ngay tới **min-heap** (`heapq`). Nếu `k`, `n` nhỏ thì quét tay / `index(min)` cũng qua.
- **Luật hòa:** biến luật thành **khóa so sánh**. Ở đây khóa là `(giá trị, chỉ số)`. Tuple so sánh từ trái sang phải, nên đừng đảo thứ tự.
- **Heap lưu "cặp" thay vì giá trị trần** khi cần biết **vị trí gốc** của phần tử (chỉ số, tên, ưu tiên phụ...).
- **`heapq` không có `peek`/cập nhật tại chỗ**: muốn đổi giá trị thì `pop` rồi `push` lại (hoặc `heapreplace`).
- **Sửa tại chỗ hay sao chép?** Bản nộp sửa thẳng `nums` rồi trả về là hợp lệ; khi tự test, truyền `nums[:]` để giữ đầu vào cho lần so sánh khác.
- **Kiểm tra `multiplier = 1`:** nó biến bài thành "không làm gì" — một trường hợp biên đơn giản để chắc thuật toán không bị kẹt.

## 15. Biến thể của bài

1. **Bài II (Medium).** `k` có thể tới `10^9`, `n` nhỏ hơn; `multiplier` có thể lớn. Không thể mô phỏng từng thao tác: sau khi mọi phần tử đã được nhân ít nhất một lần, các thao tác lặp theo **vòng** (mỗi vòng nhân từng phần tử đúng một lần theo thứ tự sắp), nên dùng **lũy thừa nhanh** `pow(multiplier, q, MOD)` và chia `k` cho `n`.
2. **Nhân phần tử lớn nhất:** chỉ đổi cặp thành `(-x, i)` (max-heap bằng cách đổi dấu).
3. **Cộng thay vì nhân:** `x + add` — heap vẫn như cũ.
4. **Trả về thứ tự các chỉ số được chọn:** ghi lại `i` mỗi thao tác.
5. **Dừng sớm:** dừng khi phần tử nhỏ nhất đạt ngưỡng `T` (số thao tác tối thiểu để mọi phần tử `>= T`) — bài "Minimum Operations to Exceed Threshold".

In [30]:
# Minh họa biến thể 2 (nhân phần tử LỚN nhất) và biến thể 4 (ghi lại chỉ số được chọn)
def get_final_state_max(nums: List[int], k: int, multiplier: int) -> List[int]:
    nums = nums[:]
    pq = [(-x, i) for i, x in enumerate(nums)]     # đổi dấu để heap nhỏ nhất thành "lớn nhất"
    heapq.heapify(pq)                              # hòa -> chỉ số nhỏ nhất vẫn thắng
    for _ in range(k):
        _, i = heapq.heappop(pq)
        nums[i] *= multiplier
        heapq.heappush(pq, (-nums[i], i))
    return nums


def chosen_indices(nums: List[int], k: int, multiplier: int) -> List[int]:
    nums = nums[:]
    pq = [(x, i) for i, x in enumerate(nums)]
    heapq.heapify(pq)
    out = []
    for _ in range(k):
        _, i = heapq.heappop(pq)
        out.append(i)
        nums[i] *= multiplier
        heapq.heappush(pq, (nums[i], i))
    return out


print(get_final_state_max([2, 1, 3, 5, 6], 3, 2))        # phần tử lớn nhất 6 ở chỉ số 4: 6 -> 12 -> 24 -> 48
print(chosen_indices([2, 1, 3, 5, 6], 5, 2))              # [1, 0, 1, 2, 0]

[2, 1, 3, 5, 48]
[1, 0, 1, 2, 0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space (thêm) | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét tay | Vòng `for` tìm chỉ số min (dùng `<`) | `O(k · n)` | `O(1)` | Hiểu đề; làm bản chuẩn đối chiếu |
| 2. `index(min)` | `nums.index(min(nums))` | `O(k · n)` | `O(1)` | Gọn nhất, đủ cho `n, k` nhỏ như đề |
| 3. Min-heap | Heap `(giá trị, chỉ số)`, pop–nhân–push | `O(n + k log n)` | `O(n)` | **Cách nộp chuẩn**, mở rộng cho `k` lớn |
| 4. List sắp + `insort` | Giữ list cặp luôn sắp | `O(n log n + k · n)` | `O(n)` | Trực quan hơn heap, khi `n` nhỏ |

Lời khuyên: nộp **Cách 3** (hoặc Cách 2 nếu muốn ngắn nhất); luôn giữ Cách 1 làm chuẩn kiểm thử.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[4, 2, 2, 7].index(min([4, 2, 2, 7]))`, `(3, 1) < (3, 2)`, `Solution().getFinalState([5], 2, 3)`, `Solution().getFinalState([3, 1, 2], 1, 4)`.

**Bài 2 (tính tay).** `nums = [3, 3, 2]`, `k = 4`, `multiplier = 2`: liệt kê mảng sau từng thao tác (nhớ luật hòa), cho kết quả cuối.

**Bài 3 (điền TODO).** Hoàn thành `get_final_state_todo(nums, k, multiplier)` theo Cách 3 (heap `(giá trị, chỉ số)`).

In [31]:
def get_final_state_todo(nums, k, multiplier):
    # TODO 1: tạo pq = [(x, i) for i, x in enumerate(nums)] rồi heapq.heapify(pq)
    # TODO 2: lặp k lần: _, i = heapq.heappop(pq)
    # TODO 3: nums[i] *= multiplier
    # TODO 4: heapq.heappush(pq, (nums[i], i))
    # TODO 5: trả về nums
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn lời giải, viết `get_final_state_sorted2(nums, k, multiplier)` dùng `min(range(n), key=lambda j: nums[j])` (mỗi thao tác) và kiểm tra nó khớp với Cách 3 trên vài test.

**Bài 5 (biến thể / mini challenge).** (a) Viết `get_final_state_add(nums, k, add)`: thay phần tử nhỏ nhất (đầu tiên khi hòa) bằng `x + add`. (b) Viết `ops_per_index(nums, k, multiplier)` trả về list `cnt` với `cnt[i]` = số lần chỉ số `i` được chọn. (c) Viết `min_ops_until_all_at_least(nums, T, multiplier)` trả về số thao tác ít nhất để **mọi** phần tử `>= T` (với `multiplier >= 2`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `index(min)` cho chỉ số đầu tiên của `2` là `1`; tuple so sánh phần tử đầu rồi tới phần tử sau; `[5]` nhân `3` hai lần là `45`; `[3,1,2]` nhân phần tử nhỏ nhất `1` thành `4`. Bài 2: mỗi lần tìm số nhỏ nhất, hòa thì chọn bên trái.
- **Mức 2 (vừa):** Bài 3: giống mục 9. Bài 4: `i = min(range(len(nums)), key=lambda j: nums[j])` trả về chỉ số nhỏ nhất, đầu tiên khi hòa.
- **Mức 3 (sâu):** Bài 5(a): chỉ đổi `*=` thành cộng. 5(b): thêm `cnt[i] += 1` mỗi thao tác. 5(c): vòng `while min(nums) < T:` — dùng heap, mỗi lần pop phần tử nhỏ nhất nhân lên và đếm; điều kiện dừng khi `heap[0][0] >= T`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert [4, 2, 2, 7].index(min([4, 2, 2, 7])) == 1
assert (3, 1) < (3, 2)
assert Solution().getFinalState([5], 2, 3) == [45]
assert Solution().getFinalState([3, 1, 2], 1, 4) == [3, 4, 2]

# Bài 2: [3,3,2] -> [3,3,4] -> [6,3,4] -> [6,6,4] -> [6,6,8]
steps = []
a = [3, 3, 2]
for _ in range(4):
    i = a.index(min(a))
    a[i] *= 2
    steps.append(a[:])
assert steps == [[3, 3, 4], [6, 3, 4], [6, 6, 4], [6, 6, 8]]
assert get_final_state_heap([3, 3, 2], 4, 2) == [6, 6, 8]


# Bài 3
def get_final_state_todo_ans(nums, k, multiplier):
    nums = nums[:]
    pq = [(x, i) for i, x in enumerate(nums)]
    heapq.heapify(pq)
    for _ in range(k):
        _, i = heapq.heappop(pq)
        nums[i] *= multiplier
        heapq.heappush(pq, (nums[i], i))
    return nums


for nums, k, multiplier, expected in examples:
    assert get_final_state_todo_ans(nums, k, multiplier) == expected


# Bài 4
def get_final_state_sorted2(nums, k, multiplier):
    nums = nums[:]
    for _ in range(k):
        i = min(range(len(nums)), key=lambda j: nums[j])
        nums[i] *= multiplier
    return nums


rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 30)
    nums = [rng.randint(1, 10) for _ in range(n)]
    k, m = rng.randint(1, 10), rng.randint(1, 5)
    assert get_final_state_sorted2(nums, k, m) == get_final_state_heap(nums, k, m)


# Bài 5
def get_final_state_add(nums, k, add):
    nums = nums[:]
    for _ in range(k):
        i = nums.index(min(nums))
        nums[i] += add
    return nums


def ops_per_index(nums, k, multiplier):
    nums = nums[:]
    cnt = [0] * len(nums)
    for _ in range(k):
        i = nums.index(min(nums))
        nums[i] *= multiplier
        cnt[i] += 1
    return cnt


def min_ops_until_all_at_least(nums, T, multiplier):
    pq = [(x, i) for i, x in enumerate(nums)]
    heapq.heapify(pq)
    ops = 0
    while pq[0][0] < T:
        x, i = heapq.heappop(pq)
        heapq.heappush(pq, (x * multiplier, i))
        ops += 1
    return ops


assert get_final_state_add([2, 1, 3], 3, 2) == [4, 5, 3]          # [2,1,3] -> [2,3,3] -> [4,3,3] -> [4,5,3]
assert ops_per_index([2, 1, 3, 5, 6], 5, 2) == [2, 2, 1, 0, 0]    # chỉ số được chọn: 1, 0, 1, 2, 0
# số 1 cần 4 lần nhân đôi (1 -> 16), số 2 cần 3 lần (2 -> 16): tổng 7
assert min_ops_until_all_at_least([1, 2], 10, 2) == 7
assert min_ops_until_all_at_least([10, 11], 10, 2) == 0
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (min-heap các cặp `(giá trị, chỉ số)`: pop – nhân – push)

```python
class Solution:
    def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]:
        pq = [(x, i) for i, x in enumerate(nums)]
        heapq.heapify(pq)
        for _ in range(k):
            _, i = heapq.heappop(pq)
            nums[i] *= multiplier
            heapq.heappush(pq, (nums[i], i))
        return nums
```

**Checklist**

- [ ] Chọn phần tử **nhỏ nhất** (không phải lớn nhất), và khi hòa chọn **chỉ số nhỏ nhất**?
- [ ] Cặp trong heap là `(giá trị, chỉ số)` — **giá trị trước**?
- [ ] Sau khi nhân có **gán lại** `nums[i]` và **`heappush` lại** cặp mới?
- [ ] Không sắp xếp làm mất vị trí gốc của mảng?
- [ ] Đã thử hai ví dụ đề và biên: `multiplier = 1`, mảng một phần tử, nhiều phần tử bằng nhau?
- [ ] Skeleton đúng `class Solution` / `def getFinalState(self, nums: List[int], k: int, multiplier: int) -> List[int]`?